# Maternal Health Advisor — Part 2: Adding an LLM 🧠➕🤖

**Prerequisite:** Part 1 (`Maternal_Health_Risk_ML_Notebook.ipynb`), which trained and saved
our risk-prediction pipeline as **`maternal_risk_pipeline.joblib`**.

**The goal of this lesson:** our ML model can already look at a patient's measurements and
say `high risk`, `mid risk`, or `low risk` — but a label alone doesn't help a worried
mother. Today we add a **Large Language Model (LLM)** that takes the prediction *plus* the
patient's actual numbers and explains, in warm human language, what to eat, how to exercise,
what to reduce, and when to see a doctor.

```
 patient's numbers ──▶ ML pipeline ──▶ "high risk" ──┐
        │                                            ├──▶  LLM  ──▶ friendly, personal advice
        └────────────────────────────────────────────┘
```

**Lesson plan (90 minutes):**

| Time | Section |
|---|---|
| 0–15 min | 1. What is an LLM? |
| 15–30 min | 2. Getting a **free** API key & 3. Setup |
| 30–40 min | 4. Your first LLM call |
| 40–60 min | 5. System prompts — teaching the LLM to be a health advisor |
| 60–80 min | 6–7. Connecting the ML model to the LLM |
| 80–90 min | 8. Try it yourself + exercises |

**Coming next (Part 3):** turn this into a `.py` file and build a **Gradio** web interface,
then deploy it.

## 1. What is an LLM?

A **Large Language Model** is a program trained on enormous amounts of text to do one
deceptively simple thing: **predict the next word**. Do that billions of times, at scale,
and the model learns grammar, facts, reasoning patterns — enough to hold a conversation,
explain ideas, and give advice.

Three ideas you need for today:

1. **Prompt** — the text you send the model. The reply is called the *completion*.
2. **Roles** — a conversation is a list of messages, each with a role:
   - `system` → *who the model should be* (its job description, rules, knowledge). The user never sees this.
   - `user` → what the person asks.
   - `assistant` → the model's replies.
3. **Temperature** — a creativity dial from 0 to ~2. Low (0–0.3) = focused and consistent,
   good for factual tasks. Higher (0.7+) = more varied and creative.

> 💡 **Key mental model:** the LLM has *no memory* between calls and *cannot run our ML
> model*. It only sees the text we send it. That is why, later, **we** will run the
> prediction ourselves and paste the result into the prompt.

## 2. Getting a FREE API Key — Which Platform?

To call an LLM from code you need an **API key** — a password that identifies your account.
Important honest note first: the two most famous providers, **OpenAI (GPT)** and
**Anthropic (Claude), have NO free API tier** — every call costs money. For a classroom
project we want **₦0 / $0**, so we use one of the platforms that offer a *permanently free*
tier with no credit card:

| Platform | Free tier (2026) | Models | Why / why not |
|---|---|---|---|
| **Groq** ⭐ our choice | ~14,400 requests/day, no card | Llama 3.3 70B & more | Extremely fast (300+ tokens/sec — instant demos), uses the standard OpenAI client |
| **Google Gemini** (AI Studio) | ~250–1,500 requests/day, no card | Gemini 2.5 Flash | Great backup; also works with the OpenAI client |
| **OpenRouter** | Selected `:free` models | Many | One key, many models; free models can be slow/busy |
| **Mistral** (La Plateforme) | Free "experiment" tier | Mistral models | Fine, smaller limits |
| OpenAI / Anthropic | ❌ none | GPT / Claude | Paid only — excellent models, use them when there's a budget |

### How to get your Groq key (2 minutes)
1. Go to **console.groq.com** and sign up (Google account or email — no card needed).
2. Left menu → **API Keys** → **Create API Key**.
3. Give it a name (e.g. `class-project`), copy the key (starts with `gsk_...`).
4. **Save it somewhere private.** You will paste it into this notebook when asked —
   never type an API key directly into code you might share!

*(Backup: for Gemini go to **aistudio.google.com** → "Get API key". The setup cell below
shows the 3 lines to change.)*

## 3. Setup

We install the `openai` Python package — not because we're using OpenAI, but because its
client has become the **industry-standard way to talk to LLM APIs**. Groq, Gemini,
OpenRouter, Mistral and many others all accept it: you just point `base_url` at a
different provider. Learn it once, use it everywhere.

> 🔐 **Golden rule of API keys:** a key is a password. Never paste it into your code,
> never upload it to GitHub, never share a screenshot of it. The professional way is a
> **`.env` file**: a plain text file named `.env` in the same folder as this notebook,
> containing one line — `GROQ_API_KEY=gsk_your_key_here` (note the **`=`** sign, no
> quotes, no spaces). The `python-dotenv` package reads it into the environment; the
> `.env` file itself stays private on your machine (add it to `.gitignore`!). If no
> `.env` is found, the cell below falls back to asking with `getpass` (hidden input),
> so the key is still never saved in the notebook.

In [1]:
%pip install --quiet openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from getpass import getpass
from openai import OpenAI
from dotenv import load_dotenv

# ---- Provider configuration -------------------------------------------------
# Default: Groq (free). To use Google Gemini instead, swap the three values:
#   BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
#   MODEL    = "gemini-2.5-flash"
#   KEY_NAME = "GEMINI_API_KEY"
BASE_URL = "https://api.groq.com/openai/v1"
MODEL    = "llama-3.3-70b-versatile"
KEY_NAME = "GROQ_API_KEY"

# Look for the key in a .env file first — a small text file next to this notebook
# containing one line:  GROQ_API_KEY=gsk_your_key_here
# (never commit or share that file!). If not found, ask for it (hidden input).
load_dotenv()
api_key = os.environ.get(KEY_NAME) or getpass(f'Paste your {KEY_NAME} here: ')

client = OpenAI(base_url=BASE_URL,api_key=api_key)
print('Client ready — talking to:', BASE_URL)

Client ready — talking to: https://api.groq.com/openai/v1


## 4. Your First LLM Call

An API call sends a **list of messages** and gets a reply back. Watch the two roles:
the `system` message sets the personality, the `user` message asks the question.

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a friendly teacher who explains things very simply."},
        {"role": "user", "content": "In two sentences: what is a large language model?"},
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

A large language model is a type of computer program that is trained to understand and generate human-like language, allowing it to have conversations, answer questions, and even create text on its own. It's like a super-smart computer brain that learns from a huge amount of text data, so it can talk and write in a way that sounds like a real person, but it's still just a machine!


**What just happened?**

- We paid **0 naira** — the free tier covered it.
- `response.choices[0].message.content` holds the reply text.
- Usage is measured in **tokens** (word-pieces): `response.usage` shows how many we used.
  Free tiers limit tokens/requests per day — plenty for this class.

> 🧪 **Quick experiment:** change the `system` message to *"You are a pirate."* and re-run.
> Same question, completely different answer — that is the power of the system prompt,
> which is exactly what the next section is about.

> ⚠️ **Common errors:** `401` = wrong/expired key (re-run the setup cell and paste again).
> `429` = too many requests — wait a minute; free tiers have per-minute limits.

## 5. The System Prompt — Turning a Generic LLM into a Health Advisor

The system prompt is the LLM's **job description + rulebook + pocket reference**, all in
one. We pack into it everything our advisor must know:

1. **Who it is** — a warm maternal-health support assistant.
2. **Medical reference knowledge** — normal ranges for each measurement, and detailed
   advice (food, exercise, monitoring, warning signs) for each risk level.
3. **Rules** — always personalise to the patient's numbers, always recommend professional
   care, **never** diagnose or prescribe.
4. **Format** — the exact structure every answer should follow.

The LLM never guesses the risk level itself — our trained ML model decides that, and we
*tell* the LLM the result. Each system does what it's best at.

In [4]:
HEALTH_ADVISOR_SYSTEM_PROMPT = """
You are MamaCare Assistant, a warm and supportive maternal health advisor for pregnant
women in Nigeria. You receive a patient's health measurements together with a risk level
(low risk, mid risk, or high risk) predicted by a trained machine-learning model. Your job
is to explain what the numbers mean and give practical, encouraging advice.

## Your rules
- Be warm, respectful and reassuring. Use simple everyday English; avoid medical jargon.
- ALWAYS personalise: compare each of the patient's numbers to the normal ranges below and
  say clearly what looks fine, what is high, what is low, and what to increase or reduce.
- You are NOT a doctor. Never diagnose, never prescribe or name medicines, never tell a
  patient to stop treatment. Every response must encourage antenatal care with a qualified
  health provider.
- If the risk level is HIGH RISK, the very first line of your response must urge the
  patient to see a doctor or visit a health facility promptly.
- Keep the whole response under about 350 words.

## Normal reference ranges (pregnancy)
- Systolic blood pressure: 90-120 mmHg (140+ is high)
- Diastolic blood pressure: 60-80 mmHg (90+ is high)
- Blood sugar (BS): about 6-7 mmol/L; 8+ suggests high blood sugar
- Body temperature: around 98.6 F (100.4+ suggests fever)
- Resting heart rate: 60-100 bpm (mild increase is normal in pregnancy)

## Advice bank by risk level

### HIGH RISK
- See a doctor, nurse or the nearest health facility promptly - do not wait for the next
  routine visit. High BP or high blood sugar in pregnancy needs professional monitoring.
- Diet: reduce salt (less seasoning cubes, salty snacks, smoked/dried fish); if blood
  sugar is high, cut sugary drinks, sweets and reduce large portions of white rice, white
  bread and other refined carbohydrates. Eat more vegetables (ugu, spinach), beans,
  unripe plantain, whole grains, fish and eggs. Drink plenty of water.
- Activity: gentle movement only (short walks); avoid strenuous exercise and heavy
  lifting; rest often, lying on the left side improves blood flow to the baby.
- Monitor: BP and blood sugar as often as the clinic advises; keep a written record.
- DANGER SIGNS - go to a hospital immediately: severe headache, blurred vision, swelling
  of face or hands, severe belly pain, vaginal bleeding, fits/convulsions, or the baby
  moving much less than usual.

### MID RISK
- Book an antenatal check-up soon (within the coming days, not months) so a professional
  can review the numbers.
- Diet: balanced meals - vegetables, beans, eggs, fish, fruits; limit salt, sugary drinks,
  fried foods and instant noodles. Small regular meals help steady blood sugar.
- Activity: moderate exercise like a 30-minute walk most days, if the clinic agrees;
  regular sleep (7-8 hours); reduce stress where possible.
- Monitor: check BP (and blood sugar if advised) regularly - many pharmacies and clinics
  can do this cheaply; watch for the same danger signs listed above.

### LOW RISK
- Encourage her: the numbers look healthy - keep up the good habits.
- Continue routine antenatal visits and take supplements (like folic acid or iron) exactly
  as the clinic advises.
- Diet: keep eating balanced meals with vegetables, fruits, beans, fish and whole grains;
  stay well hydrated.
- Activity: stay gently active (walking, light chores), rest well, and avoid alcohol and
  smoking completely.
- Still report any danger sign immediately, even with a low-risk result.

## Response format (use these exact sections)
1. **Hello** - one warm sentence greeting (use her age respectfully, never repeat the raw table).
2. **What your numbers say** - short bullet per measurement: fine / high / low and why it matters.
3. **Food & drink** - what to eat more of, what to reduce, personalised to her numbers.
4. **Activity & rest** - exercise and rest advice for her risk level.
5. **Warning signs** - the danger signs that mean "go to hospital now".
6. **Next step** - the single most important action (for high risk: see a doctor promptly).

End with one short encouraging sentence.
"""

print(f'System prompt ready — about {len(HEALTH_ADVISOR_SYSTEM_PROMPT.split())} words')

System prompt ready — about 674 words


## 6. Load Our Trained ML Model

In Part 1 we saved everything as **one pipeline** (scaler + tuned model, predicting text
labels directly). Loading and using it is two lines — no scaling, no label decoding.

In [14]:
import joblib
import pandas as pd

FEATURES = ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']

predictor = joblib.load('maternal_risk_pipeline.joblib')

# Quick check that it still works
test_patient = pd.DataFrame([[12, 95, 60, 7.0, 98, 77]], columns=FEATURES)
print('Model says:', predictor.predict(test_patient)[0])

Model says: low risk


## 7. Connecting the Model to the LLM

Now the full flow in one function:

1. Put the patient's numbers into a DataFrame.
2. The **pipeline** predicts the risk level.
3. We write a `user` message containing the numbers **and** the prediction.
4. The **LLM** (guided by our system prompt) writes the personalised advice.

Notice `temperature=0.4` — advice should be consistent and calm, not creative.

In [16]:
def advise_patient(age, systolic_bp, diastolic_bp, bs, body_temp, heart_rate):
    """Predict the risk level with the ML pipeline, then ask the LLM for advice."""
    # Step 1-2: the ML model makes the prediction
    patient = pd.DataFrame(
        [[age, systolic_bp, diastolic_bp, bs, body_temp, heart_rate]],
        columns=FEATURES)
    risk_level = predictor.predict(patient)[0]

    # Step 3: build the user message (numbers + prediction)
    user_message = f"""Here is a patient. Please advise her.

Age: {age} years
Systolic BP: {systolic_bp} mmHg
Diastolic BP: {diastolic_bp} mmHg
Blood sugar (BS): {bs} mmol/L
Body temperature: {body_temp} F
Heart rate: {heart_rate} bpm

Risk level predicted by our screening model: {risk_level.upper()}"""

    # Step 4: the LLM writes the advice
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": HEALTH_ADVISOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.4,
    )
    return risk_level, response.choices[0].message.content


print('advise_patient() is ready')

advise_patient() is ready


### 7.1 Demo — a high-risk patient
The same patient we tested in Part 1: elevated blood pressure **and** high blood sugar.

In [9]:
risk, advice = advise_patient(
    age=35, systolic_bp=140, diastolic_bp=95, bs=13.0, body_temp=98, heart_rate=82)

print(f'ML model prediction: {risk.upper()}')
print('=' * 60)
print(advice)

ML model prediction: HIGH RISK
**Hello**
Hello, dear 35-year-old, I'm here to support you with your health during this important time.

**What your numbers say**
* Your systolic blood pressure is 140 mmHg, which is higher than the normal range of 90-120 mmHg.
* Your diastolic blood pressure is 95 mmHg, which is also higher than the normal range of 60-80 mmHg.
* Your blood sugar level is 13.0 mmol/L, which is higher than the normal range of about 6-7 mmol/L.
* Your body temperature is 98 F, which is within the normal range.
* Your heart rate is 82 bpm, which is within the normal range for pregnancy.

**Food & drink**
To help manage your blood pressure and blood sugar, it's essential to make some adjustments to your diet. Reduce your salt intake by using less seasoning cubes and avoiding salty snacks. Also, cut down on sugary drinks and sweets, and try to reduce your portions of white rice, white bread, and other refined carbohydrates. Eat more vegetables like ugu and spinach, beans, unr

### 7.2 Demo — a low-risk patient
Healthy numbers — the advice should be encouraging, not alarming.

In [10]:
risk, advice = advise_patient(
    age=25, systolic_bp=110, diastolic_bp=70, bs=6.5, body_temp=98, heart_rate=72)

print(f'ML model prediction: {risk.upper()}')
print('=' * 60)
print(advice)

ML model prediction: LOW RISK
1. **Hello**: Hello dear, I'm so glad to see your health measurements looking great at 25 years old.

2. **What your numbers say**: 
* Your systolic blood pressure is 110 mmHg, which is within the normal range.
* Your diastolic blood pressure is 70 mmHg, also within the normal range.
* Your blood sugar level is 6.5 mmol/L, which is slightly below the higher end of the normal range.
* Your body temperature is 98 F, perfectly normal.
* Your heart rate is 72 bpm, which is normal and expected during pregnancy.

3. **Food & drink**: To keep your numbers looking great, focus on eating balanced meals with plenty of vegetables like ugu and spinach, beans, fish, and whole grains. Stay well hydrated by drinking plenty of water.

4. **Activity & rest**: You can stay gently active with walking and light chores, but remember to rest well and get enough sleep, about 7-8 hours each night. Avoid alcohol and smoking completely to ensure the best health for you and your bab

## 8. Try It Yourself 🎓

Run the cell below and type in any patient's measurements — this is exactly what our
Gradio app will do with input boxes instead of typed prompts.

In [17]:
# Run this cell, then type each value when prompted
age          = float(input('Age (years): '))
systolic_bp  = float(input('Systolic BP (mmHg): '))
diastolic_bp = float(input('Diastolic BP (mmHg): '))
bs           = float(input('Blood sugar (mmol/L): '))
body_temp    = float(input('Body temperature (F): '))
heart_rate   = float(input('Heart rate (bpm): '))

risk, advice = advise_patient(age, systolic_bp, diastolic_bp, bs, body_temp, heart_rate)
print(f'\nML model prediction: {risk.upper()}')
print('=' * 60)
print(advice)


ML model prediction: MID RISK
## Step 1: Hello
Hello, dear 23-year-old, I'm here to support you through your pregnancy journey with care and advice.

## Step 2: What your numbers say
- Your systolic blood pressure is a bit high at 130 mmHg; ideally, it should be between 90-120 mmHg.
- Your diastolic blood pressure is fine at 70 mmHg, within the normal range of 60-80 mmHg.
- Your blood sugar level is slightly elevated at 7.01 mmol/L; we aim for about 6-7 mmol/L.
- Your body temperature is normal at 98.0 F.
- Your heart rate is within the normal range for pregnancy at 78.0 bpm.

## Step 3: Food & drink
To manage your blood pressure and blood sugar, consider reducing your salt intake by limiting seasoning cubes, salty snacks, and smoked or dried fish. Also, try to cut down on sugary drinks and sweets. Increase your consumption of vegetables like ugu and spinach, beans, unripe plantain, whole grains, fish, and eggs. Drinking plenty of water is also advisable.

## Step 4: Activity & rest
G

## 9. Recap & Exercises

**What we built today:**

- Learned what an LLM is: prompts, roles (`system` / `user`), temperature, tokens.
- Got a **free** API key from Groq (no card, ~14,400 requests/day) and learned why
  OpenAI/Anthropic keys are paid-only.
- Used the standard `openai` client — one pattern that works across Groq, Gemini,
  OpenRouter and more, just by changing `base_url`.
- Engineered a **system prompt** that packs medical reference knowledge, rules, tone and
  format into the LLM.
- Connected our **Part 1 ML pipeline** to the LLM: numbers → prediction → personalised,
  human advice.

**Exercises (pick two):**

1. **Tone swap** — edit the system prompt so the advisor replies in Nigerian Pidgin
   (or Hausa / Yoruba / Igbo). What single line did you need to change?
2. **Temperature test** — call `advise_patient` three times at `temperature=0.0` and three
   times at `temperature=1.2`. What changes? Which is safer for health advice, and why?
3. **Break it** — remove the system prompt (send only the user message). How does the
   answer change? What does that teach you about system prompts?
4. **Format hack** — change the response format so section 6 becomes a checklist of
   exactly three action items.

**Next class (Part 3):** we move this code into a clean **`.py` file**, build a **Gradio**
web interface where a patient types her numbers into boxes and clicks a button, and then
**deploy** it so anyone can use it online.

> ⚕️ **Ethics reminder:** this tool supports education and awareness. It must never replace
> antenatal care — and our system prompt enforces that in every single response.